# Установка библиотек. Импорты

In [1]:
! pip install bitsandbytes evaluate
# ! pip install flash-attn --no-build-isolation
# ! pip install --upgrade transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import transformers
from transformers import(
AutoTokenizer,
AutoModel,
BitsAndBytesConfig,
TrainingArguments,
DataCollatorWithPadding,
Trainer
)
import datasets
import bitsandbytes as bnb
import accelerate
import peft
import sklearn
from huggingface_hub import login
import evaluate
import seaborn as sns

In [3]:
# from transformers.cache_utils import DynamicCache
# import pprint
# pprint.pprint(dir(DynamicCache))

# print(f"Transformers: {transformers.__version__}")
print(f"BitsAndBytes: {bnb.__version__}")
# print(f"PyTorch: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")

BitsAndBytes: 0.49.2


In [4]:
print(f"Количество GPU доступно: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Количество GPU доступно: 1
GPU 0: Tesla T4


In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
if hf_token:
    print('авторизация на Hugging Face...')
    login(hf_token)
    print('авторизация на Hugging Face прошла успешно!')
else:
    print('Токен не найден')

авторизация на Hugging Face...
авторизация на Hugging Face прошла успешно!


In [6]:
model_name = "microsoft/Phi-3-mini-4k-instruct"
num_labels = 2
max_length = 256

# Загрузка датасета и его предобработка

#

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                         trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

# tokenizer.add_special_tokens({'additional_special_tokens':['[CLS]'],
#                               'pad_token' : '[PAD]',
#                              })
print(len(tokenizer))

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

32011


In [8]:
import typing
from typing import Dict, Any
label2id = {'ai': 0, 'human': 1}
def tokenize_examples(examples: Dict[str, Any]) -> Dict[str, Any]:
    """
    Функция токенизации
    --------------------------
    Parameters:
    example : datasets.DatasetDict
        датасет из библиотеки HF Datasets

     Returns:
    --------
    datasets.DatasetDict
        Токенизированный датасет
    """
    result = tokenizer(examples['text'], truncation = True, max_length = 256,)
    result['label'] = [label2id[l] for l in examples['label']]
    return result

In [9]:
ds = datasets.load_dataset("iitolstykh/LLMTrace_classification")

README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/1.40G [00:00<?, ?B/s]

valid.jsonl:   0%|          | 0.00/292M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/318M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/411440 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/86696 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/90950 [00:00<?, ? examples/s]

In [10]:
ds = ds.map(tokenize_examples, batched=True, )

Map:   0%|          | 0/411440 [00:00<?, ? examples/s]

Map:   0%|          | 0/86696 [00:00<?, ? examples/s]

Map:   0%|          | 0/90950 [00:00<?, ? examples/s]

# Загрузка модели. QLoRa.

In [11]:
# # конфиг для квантования - понадобится, если обучать тяжелую модель
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type='nf4',
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

In [12]:
# del model
# del classificator
# torch.cuda.empty_cache()
# gc.collect

In [13]:
# ! nvidia-smi

In [14]:
model = AutoModel.from_pretrained(model_name,
                                            torch_dtype=torch.float16,
                                            device_map='auto',
                                             # device_map={"": 0},
                                            output_hidden_states=False, # hidden state векторы для дальнейшей классификации
                                            trust_remote_code=False, 
                                            # quantization_config=bnb_config, # конфиг для квантования модели
                                            low_cpu_mem_usage=True,
                                            attn_implementation='sdpa',
                                            )

torch.cuda.empty_cache()
# model.resize_token_embeddings(len(tokenizer)) # делайм ресайз эмбеддингов, так как добавили новые токены в словарь

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Phi3Model LOAD REPORT from: microsoft/Phi-3-mini-4k-instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
print(model)

Phi3Model(
  (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
  (layers): ModuleList(
    (0-31): 32 x Phi3DecoderLayer(
      (self_attn): Phi3Attention(
        (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
      )
      (mlp): Phi3MLP(
        (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
        (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
        (activation_fn): SiLUActivation()
      )
      (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
      (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
      (resid_attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (norm): Phi3RMSNorm((3072,), eps=1e-05)
  (rotary_emb): Phi3RotaryEmbedding()
)


In [16]:
# конфиг для LoRa
lora_config = peft.LoraConfig(task_type='FEATURE_EXTRACTION',
                              r=8,
                              lora_alpha=16,
                              lora_dropout = 0.1,
                              target_modules='all-linear',
                              bias='none',
                              # modules_to_save = ['embed_tokens'],
                              # ensure_weight_tying = True # гарантируем, что embed_tokens и lm_head будут одинаковыми
                             )

In [17]:
if isinstance(model, peft.PeftModel):
    model = model.unload()
    print('старый PEFT конфиг удален. Загружена базовая модель.')

# peft.prepare_model_for_kbit_training(model, use_gradient_checkpointing=True) # подготовка к обучению с квантизацией
model.enable_input_require_grads() # включаем принудительно нужные нам градиенты
# применяем конфиг к модели
model = peft.get_peft_model(model=model, peft_config=lora_config)
# model.gradient_checkpoint_enable() # храним градиенты только для контрольных точек
print('После применения PEFT-конфига: ')
model.print_trainable_parameters()

После применения PEFT-конфига: 
trainable params: 12,582,912 || all params: 3,735,161,856 || trainable%: 0.3369


In [18]:
type(tokenizer)

transformers.tokenization_utils_tokenizers.TokenizersBackend

In [19]:
model.config.hidden_size

3072

# Класс модели и её иницализация

In [20]:
class LLM_Classificator(nn.Module):
    """
    Классификатор текста на основе LLM
    """
    
    def __init__(self, base_model, tokenizer, num_labels=2):
        super().__init__()
        self.base_model = base_model
        self.tokenizer = tokenizer
        self.num_labels = num_labels
        # self.cls_tok_id = tokenizer.convert_tokens_to_ids('[CLS]')
        self.classification_head = nn.Linear(base_model.config.hidden_size, 
                                             num_labels, 
                                             # dtype=torch.float16
                                            ).to(base_model.device)
        self.loss_func = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        # print(f"DEBUG: keys in batch: {kwargs.keys()}")
        outputs = self.base_model.model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=False,
            use_cache=False,
            **kwargs
        )
        # берем последнее состояние контекста
        last_hidden_state = outputs.last_hidden_state
        # извлекаем эмбеддинг для последнего токена
        last_token_embed = last_hidden_state[:, -1, :]
        last_token_embed = last_token_embed.to(self.classification_head.weight.dtype)
        # print(last_token_embed.device, last_token_embed.dtype)
        # прогоняем эмбеддинг последнего токена через слой классификатора
        logits = self.classification_head(last_token_embed)
        # loss = None

        if labels is not None:
            # loss_func = nn.CrossEntropyLoss()
            loss = self.loss_func(logits.view(-1, self.num_labels), labels.view(-1)) # считаем loss score для батча
        
        return {'logits': logits, 'loss': loss}

    @property
    def device(self):
        return next(self.parameters()).device

In [21]:
# тест класса 
classificator = LLM_Classificator(base_model=model, tokenizer=tokenizer)
# print(classificator)
# classificator.to('cuda')
classificator.eval()
inputs = tokenizer('Это тестовый пример работы', return_tensors='pt')
inputs.to('cuda')
classificator(**inputs, labels=torch.tensor([1]).to('cuda'))

{'logits': tensor([[-0.8599,  0.5016]], device='cuda:0', grad_fn=<AddmmBackward0>),
 'loss': tensor(0.2281, device='cuda:0', grad_fn=<NllLossBackward0>)}

In [22]:
# import torch

# # Примеры текстов разной длины
# test_texts = [
#     "Hello world",
#     "Artificial intelligence is transforming our world",
#     "Short",
# ]

# print("=" * 80)
# print("ПРОВЕРКА ТОКЕНИЗАТОРА (pad_side='left')")
# print("=" * 80)

# # Токенизация с теми же параметрами, что будут при обучении
# tokenized = tokenizer(
#     test_texts,
#     truncation=True,
#     padding='max_length',
#     max_length=max_length,  # Ваш max_length (например, 256)
#     padding_side='left',
#     return_tensors='pt'
# )

# print(f"\n📊 Параметры:")
# print(f"  max_length: {max_length}")
# print(f"  pad_token_id: {tokenizer.pad_token_id}")
# print(f"  eos_token_id: {tokenizer.eos_token_id}")
# print(f"  pad_side: {tokenizer.padding_side}")

# print("\n" + "=" * 80)

# for i, text in enumerate(test_texts):
#     print(f"\n📝 Текст {i+1}: '{text}'")
#     print("-" * 80)
    
#     input_ids = tokenized.input_ids[i]
#     attention_mask = tokenized.attention_mask[i]
    
#     # Найдем позиции
#     non_padding_positions = (attention_mask == 1).nonzero(as_tuple=True)[0]
#     first_real_token = non_padding_positions[0].item()
#     last_real_token = non_padding_positions[-1].item()
    
#     print(f"  Длина последовательности: {len(input_ids)}")
#     print(f"  Первый реальный токен на позиции: {first_real_token}")
#     print(f"  Последний реальный токен на позиции: {last_real_token}")
    
#     # Что на последней позиции (-1)?
#     last_token_id = input_ids[-1].item()
#     last_token_decoded = tokenizer.decode([last_token_id])
    
#     print(f"\n  🔍 ПОСЛЕДНИЙ ТОКЕН (позиция -1):")
#     print(f"     ID: {last_token_id}")
#     print(f"     Текст: '{last_token_decoded}'")
    
#     # Покажем структуру последовательности
#     print(f"\n  📈 Структура последовательности:")
    
#     # Покажем первые 5 и последние 5 токенов
#     print(f"     Начало: {input_ids[:5].tolist()}")
#     print(f"     Конец:  {input_ids[-5:].tolist()}")
    
#     # Декодируем для наглядности
#     print(f"\n     Начало (текст): {tokenizer.decode(input_ids[:5])}")
#     print(f"     Конец (текст):  {tokenizer.decode(input_ids[-5:])}")
    
#     # Визуализация
#     print(f"\n  🎯 Визуализация:")
#     seq_repr = []
#     for j, (tok_id, mask) in enumerate(zip(input_ids, attention_mask)):
#         if mask == 0:
#             seq_repr.append("[PAD]")
#         else:
#             tok_str = tokenizer.decode([tok_id])
#             seq_repr.append(f"{tok_str}")
    
#     # Сожмем для отображения
#     if len(seq_repr) > 20:
#         display_seq = seq_repr[:5] + ["..."] + seq_repr[-5:]
#         print(f"     {' | '.join(display_seq)}")
#     else:
#         print(f"     {' | '.join(seq_repr)}")
    
#     # Проверка: последний токен это паддинг или реальный текст?
#     print(f"\n  ✅ ПРОВЕРКА:")
#     if attention_mask[-1].item() == 1:
#         print(f"     ✓ Последний токен — РЕАЛЬНЫЙ (не паддинг)")
#         print(f"     ✓ Можно безопасно использовать hidden_states[:, -1, :]")
#     else:
#         print(f"     ⚠ Последний токен — ПАДДИНГ!")
#         print(f"     ⚠ Нужно искать последний не-паддинг токен!")
    
#     print("\n" + "=" * 80)

# # 📊 Сводная статистика
# print("\n📊 СВОДНАЯ СТАТИСТИКА:")
# print(f"  Всего примеров: {len(test_texts)}")
# print(f"  У всех последний токен реальный: {all(tokenized.attention_mask[:, -1] == 1)}")

In [23]:
print(classificator.device)

cuda:0


In [24]:
devices = {param.device for param in classificator.parameters()}
print(f"Уникальные устройства в модели: {devices}")
# classificator.to('cuda')
# 3. Детальная проверка (если вдруг что-то осталось на CPU)
if 'cpu' in str(devices):
    for name, param in classificator.named_parameters():
        if 'cpu' in str(param.device):
            print(f"Слой {name} остался на CPU!")
else:
    print("✅ Все части модели успешно перенесены на CUDA.")

Уникальные устройства в модели: {device(type='cuda', index=0)}
✅ Все части модели успешно перенесены на CUDA.


# Метрики классификации

In [25]:
accuracy = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall = evaluate.load('recall')
f1 = evaluate.load('f1')
metrics_func = [accuracy, precision, recall, f1]
def compute_metrics(eval_pred):
    """
    Вычисляет метрики по предсказанию модели и меткам
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    result = dict()
    for func in metrics_func:
        if func is accuracy:
            result.update(func.compute(predictions=preds, references=labels))
        else:   
            result.update(func.compute(predictions=preds, references=labels, pos_label=0))
    return result

# Обучение

In [26]:
from transformers import TrainerCallback

class DebugCallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kwargs):
        if state.global_step % 500 == 0:  
            print(f"[DEBUG] Начало шага {state.global_step}")

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 500 == 0:
            print(f"[DEBUG] Завершён шаг {state.global_step}")

    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"[DEBUG] Начало эпохи {state.epoch}")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"[DEBUG] Завершена эпоха {state.epoch}")
        print(f"    Метрики: {state.log_history[-1] if state.log_history else 'нет'}")

In [27]:
demo_train = ds['train'].shuffle(seed=42).select(range(64)).select_columns(['input_ids', 'attention_mask', 'label'])
demo_valid = ds['validation'].shuffle(seed=42).select(range(64)).select_columns(['input_ids', 'attention_mask', 'label'])
# demo_test = ds['test'].shuffle(seed=42).select(range(256)).select_columns(['input_ids', 'attention_mask', 'label'])

demo_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
demo_valid.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
# demo_test.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# print(demo_train, demo_valid, demo_test)

train_set = ds['train'].shuffle(seed=42).select_columns(['input_ids', 'attention_mask', 'label'])
valid_set = ds['validation'].shuffle(seed=42).select(range(5000)).select_columns(['input_ids', 'attention_mask', 'label'])
train_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
valid_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
# test_set = ds['test'].select_columns(['input_ids', 'attention_mask', 'label'])
# print(train_set, valid_set, test_set)

In [28]:
# print(demo_train['label'])
# classificator.gradient_checkpointing_enable()
from transformers import EarlyStoppingCallback

In [29]:
training_args = TrainingArguments(
    output_dir='/kaggle/working/phi3_finetuned',
    do_train = True,
    do_eval = True,
    # eval_strategy='epoch',
    eval_strategy='steps',
    # eval_steps=10,
    eval_steps=500,
    # save_strategy='epoch',
    save_strategy='steps',
    # save_strategy='no',
    save_steps=2500,
    # save_steps=1000,
    # max_steps=32,
    max_steps=5000,
    save_total_limit=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    optim="paged_adamw_8bit",
    # gradient_checkpointing=True,
    # num_train_epochs=1,
    learning_rate=2e-5,
    fp16=True,
    dataloader_num_workers = 4,
    dataloader_pin_memory=True,
    log_level='debug',
    disable_tqdm=False,
    metric_for_best_model = 'f1',
    greater_is_better=True,
    load_best_model_at_end = True,
    report_to="none",
    logging_dir = '/kaggle/working/logs',
    logging_steps = 250,
    group_by_length=True,
    use_cache=False,
    remove_unused_columns=False
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer,)

trainer = Trainer(
    model = classificator,
    args = training_args,
    data_collator = data_collator,
    train_dataset = train_set,
    eval_dataset = valid_set,
    # train_dataset = demo_train,
    # eval_dataset = demo_valid,
    compute_metrics = compute_metrics,
    # callbacks=[DebugCallback()],
    # callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
    
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
max_steps is given, it will override any value given in num_train_epochs


In [30]:
# torch.autograd.set_detect_anomaly(True)
# transformers.utils.logging.set_verbosity_debug()
classificator.train()
trainer.train()

Currently training with a batch size of: 4
Detected bitsandbytes version: 0.49.2
skipped Embedding(32064, 3072, padding_idx=32000): 93.9375M params
bitsandbytes: will optimize Embedding(32064, 3072, padding_idx=32000) in fp32
skipped: 93.9375M params
***** Running training *****
  Num examples = 411,440
  Num Epochs = 1
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 2
  Total optimization steps = 5,000
  Number of trainable parameters = 12,589,058


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,0.760108,0.294884,0.886000,0.916667,0.888176,0.902196
1000,0.635997,0.331266,0.918400,0.982602,0.877703,0.927195
1500,0.592109,0.266158,0.930600,0.955385,0.926014,0.940470
2000,0.526320,0.253230,0.936400,0.958362,0.933108,0.945567
2500,0.546041,0.235971,0.939000,0.976661,0.918919,0.946910
3000,0.471871,0.226254,0.942000,0.974414,0.926351,0.949775
3500,0.475799,0.243529,0.941000,0.976063,0.922973,0.948776
4000,0.505101,0.232116,0.944000,0.976868,0.927365,0.951473
4500,0.519618,0.226824,0.943800,0.968193,0.935811,0.951727
5000,0.431129,0.226118,0.945200,0.969580,0.936824,0.952921



***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2
Saving model checkpoint to /kaggle/working/phi3_finetuned/checkpoint-2500
Trainer.model is not a `PreTrainedModel`, only saving its state dict.
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
chat template saved in /kaggle/working/phi3_finetuned/checkpoint-2500/chat_template.jinja
tokenizer config file saved in /kaggle/working/phi3_finetuned/checkpoint-2500/tokenizer_config.json

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Running Evaluation *****
  Num examples = 5000
  Batch size = 2

***** Run

TrainOutput(global_step=5000, training_loss=0.5650077545166016, metrics={'train_runtime': 14513.3118, 'train_samples_per_second': 2.756, 'train_steps_per_second': 0.345, 'total_flos': 0.0, 'train_loss': 0.5650077545166016, 'epoch': 0.09721952167995333})

In [31]:
import json

with open('/kaggle/working/trainer_state_manual.json', 'w') as f:
    json.dump(trainer.state.log_history, f, indent=4)

print("История обучения сохранена в trainer_state_manual.json")

История обучения сохранена в trainer_state_manual.json


In [32]:
import shutil
# 1. Путь к папке с чекпоинтами
checkpoint_dir = '/kaggle/working/phi3_finetuned'

# 2. Удаляем все промежуточные данные обучения
if os.path.exists(checkpoint_dir):
    print(f"Удаляю временные чекпоинты из {checkpoint_dir}...")
    shutil.rmtree(checkpoint_dir)
# shutil.rmtree('/kaggle/working/phi3_final_merged')

# if os.path.exists('/kaggle/working/logs'):
    # shutil.rmtree('/kaggle/working/logs')

print("Место освобождено. Пробую сохранить модель снова...")

Удаляю временные чекпоинты из /kaggle/working/phi3_finetuned...
Место освобождено. Пробую сохранить модель снова...


In [33]:
trainer.model

LLM_Classificator(
  (base_model): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [34]:
import gc
gc.collect()
torch.cuda.empty_cache()

try:
    print("Достаем PEFT-модель из обертки и мержим...")
    peft_model = trainer.model.base_model 
    merged_model = peft_model.merge_and_unload()
    print("Сохранение монолитной модели...")
    merged_model.save_pretrained("/kaggle/working/phi3_final_merged")
    tokenizer.save_pretrained("/kaggle/working/phi3_final_merged")
    print("Сохранение головы классификатора")
    torch.save(trainer.model.classification_head.state_dict(), "/kaggle/working/phi3_final_merged/classification_head.pth")
    
    print("Успешно сохранено!")

except AttributeError:
    print(" Не нашел атрибут модели. Проверь, как называется переменная внутри LLM_Classificator (self.????)")
except Exception as e:
    print(f"Ошибка: {e}")

Configuration saved in /kaggle/working/phi3_final_merged/config.json


Достаем PEFT-модель из обертки и мержим...
Сохранение монолитной модели...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/phi3_final_merged/model.safetensors
chat template saved in /kaggle/working/phi3_final_merged/chat_template.jinja
tokenizer config file saved in /kaggle/working/phi3_final_merged/tokenizer_config.json


Сохранение головы классификатора
Успешно сохранено!
